In [1]:
# ============================================================
# Step 1 — Imports & Configuration
# SupportMind AI — IT Helpdesk Sentiment Analysis
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# Reproducibility
RANDOM_STATE = 42

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\danie\supportmind-ai")

DATA_PATH = (
    PROJECT_ROOT
    / "rawsupportminddata"
    / "IT-helpdesk-synthetic-ticketshg.csv"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "backend"
    / "app"
    / "nlp"
    / "sentimental_help_analysis"
    / "models"
    / "supportmind_it_sentiment"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Sentiment configuration
# ------------------------------------------------------------

TEXT_COLUMN = "description"

SENTIMENT_LABELS = [
    "Negative",
    "Neutral",
    "Positive",
]

print("Imports and configuration loaded successfully.")
print(f"Dataset path : {DATA_PATH}")
print(f"Model path   : {MODEL_DIR}")
print(f"Text column  : {TEXT_COLUMN}")
print(f"Labels       : {SENTIMENT_LABELS}")

Imports and configuration loaded successfully.
Dataset path : C:\Users\danie\supportmind-ai\rawsupportminddata\IT-helpdesk-synthetic-ticketshg.csv
Model path   : C:\Users\danie\supportmind-ai\backend\app\nlp\sentimental_help_analysis\models\supportmind_it_sentiment
Text column  : description
Labels       : ['Negative', 'Neutral', 'Positive']


In [2]:
# ============================================================
# Step 2 — Load & Verify Dataset
# ============================================================

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")
print(f"\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nDescription preview:")
display(
    df[[TEXT_COLUMN]]
    .head(5)
)

Dataset loaded successfully.
Shape: (500, 7)

Columns:
['id', 'subject', 'description', 'priority', 'category', 'createdAt', 'requesterEmail']

Missing values:
id                0
subject           0
description       0
priority          0
category          0
createdAt         0
requesterEmail    0
dtype: int64

Description preview:


,description
0,Hey IT! Our network printer keeps disconnectin...
1,This is a follow-up to your previous request #...
2,Hey team! :wave: We're experiencing some inter...
3,We need to set up a new Google Calendar for te...
4,Jordan Smith is part of the Project Management...


In [3]:
# ============================================================
# Step 3 — Sentiment-Specific Preprocessing
# ============================================================

import re


def clean_sentiment_text(text: str) -> str:
    """
    Light preprocessing for sentiment analysis.

    Preserve sentiment-bearing words and punctuation.
    Remove only obvious technical noise.
    """

    text = str(text).strip()

    # Remove email addresses
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " ",
        text,
    )

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # Normalize repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["clean_text"] = df[TEXT_COLUMN].apply(clean_sentiment_text)

print("Sentiment preprocessing completed.")

print(f"\nOriginal descriptions : {df[TEXT_COLUMN].notna().sum():,}")
print(f"Clean descriptions    : {df['clean_text'].notna().sum():,}")
print(f"Empty clean texts     : {(df['clean_text'] == '').sum():,}")

print("\nSample cleaned descriptions:")
display(
    df[[TEXT_COLUMN, "clean_text"]].head(10)
)

Sentiment preprocessing completed.

Original descriptions : 500
Clean descriptions    : 500
Empty clean texts     : 0

Sample cleaned descriptions:


,description,clean_text
0,Hey IT! Our network printer keeps disconnectin...,Hey IT! Our network printer keeps disconnectin...
1,This is a follow-up to your previous request #...,This is a follow-up to your previous request #...
2,Hey team! :wave: We're experiencing some inter...,Hey team! :wave: We're experiencing some inter...
3,We need to set up a new Google Calendar for te...,We need to set up a new Google Calendar for te...
4,Jordan Smith is part of the Project Management...,Jordan Smith is part of the Project Management...
5,Email client fails to sync with server due to ...,Email client fails to sync with server due to ...
6,Notification from Acme IT\n\nAcme IT Image\n\n...,Notification from Acme IT Acme IT Image This A...
7,I am unable to access my email account as I ha...,I am unable to access my email account as I ha...
8,"Hello IT Team, I've been trying to access and ...","Hello IT Team, I've been trying to access and ..."
9,"Hello IT Team, I've noticed that the group pol...","Hello IT Team, I've noticed that the group pol..."


In [4]:
# ============================================================
# Step 4A — Pretrained Sentiment Teacher
# ============================================================

from transformers import pipeline

sentiment_teacher = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
)

print("Pretrained sentiment teacher loaded successfully.")

c:\Users\danie\anaconda3\envs\supportmind\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\danie\anaconda3\envs\supportmind\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\danie\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment-latest. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an a

Pretrained sentiment teacher loaded successfully.


In [5]:
# ============================================================
# Step 4B — Teacher Sanity Check
# ============================================================

sample_texts = df["clean_text"].head(10).tolist()

teacher_results = sentiment_teacher(
    sample_texts,
    truncation=True,
    max_length=512,
)

sentiment_check = pd.DataFrame(
    {
        "description": sample_texts,
        "label": [result["label"] for result in teacher_results],
        "confidence": [result["score"] for result in teacher_results],
    }
)

display(sentiment_check)

,description,label,confidence
0,Hey IT! Our network printer keeps disconnectin...,negative,0.864992
1,This is a follow-up to your previous request #...,negative,0.644089
2,Hey team! :wave: We're experiencing some inter...,negative,0.571592
3,We need to set up a new Google Calendar for te...,neutral,0.897824
4,Jordan Smith is part of the Project Management...,neutral,0.878147
5,Email client fails to sync with server due to ...,negative,0.804562
6,Notification from Acme IT Acme IT Image This A...,neutral,0.851358
7,I am unable to access my email account as I ha...,negative,0.645146
8,"Hello IT Team, I've been trying to access and ...",neutral,0.505374
9,"Hello IT Team, I've noticed that the group pol...",negative,0.696259


In [6]:
# ============================================================
# Step 5 — Generate Sentiment Pseudo-Labels
# ============================================================

texts = df["clean_text"].tolist()

teacher_results = sentiment_teacher(
    texts,
    truncation=True,
    max_length=512,
    batch_size=16,
)

df["sentiment_label"] = [
    result["label"].capitalize()
    for result in teacher_results
]

df["sentiment_confidence"] = [
    result["score"]
    for result in teacher_results
]

print("Pseudo-label generation completed.")
print(f"Rows labelled: {len(df):,}")

print("\nSentiment distribution:")
display(
    df["sentiment_label"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

print("\nConfidence statistics:")
display(
    df["sentiment_confidence"].describe()
)

Pseudo-label generation completed.
Rows labelled: 500

Sentiment distribution:


,sentiment,count
0,Neutral,246
1,Negative,215
2,Positive,39



Confidence statistics:


count    500.000000
mean       0.716990
std        0.121571
min        0.448063
25%        0.616208
50%        0.728839
75%        0.815502
max        0.951049
Name: sentiment_confidence, dtype: float64

In [7]:
# ============================================================
# Step 6 — Pseudo-Label Quality Inspection
# ============================================================

# Highest-confidence examples from each sentiment
for sentiment in ["Negative", "Neutral", "Positive"]:
    print(f"\n{'=' * 70}")
    print(f"{sentiment.upper()} — HIGH CONFIDENCE EXAMPLES")
    print(f"{'=' * 70}")

    examples = (
        df[df["sentiment_label"] == sentiment]
        .sort_values("sentiment_confidence", ascending=False)
        [["sentiment_label", "sentiment_confidence", "description"]]
        .head(5)
    )

    display(examples)


# Lowest-confidence examples overall
print(f"\n{'=' * 70}")
print("LOW-CONFIDENCE EXAMPLES")
print(f"{'=' * 70}")

low_confidence = (
    df.sort_values("sentiment_confidence")
    [["sentiment_label", "sentiment_confidence", "description"]]
    .head(15)
)

display(low_confidence)


NEGATIVE — HIGH CONFIDENCE EXAMPLES


,sentiment_label,sentiment_confidence,description
78,Negative,0.884424,"Hello, I'm experiencing persistent issues with..."
319,Negative,0.882513,"Hey @IT Support, I'm experiencing persistent i..."
244,Negative,0.877394,"Hi IT Team, I'm encountering persistent issues..."
476,Negative,0.871814,"Hello IT Team, I'm experiencing persistent iss..."
265,Negative,0.871631,I'm experiencing persistent issues with our en...



NEUTRAL — HIGH CONFIDENCE EXAMPLES


,sentiment_label,sentiment_confidence,description
33,Neutral,0.940985,Firewall Configuration Request\n\nAccount Acm...
194,Neutral,0.933667,Ticket for Jamie. Reached out via email today ...
255,Neutral,0.933582,How do I access the shared network drive from ...
246,Neutral,0.932709,This is a follow-up to your previous request #...
242,Neutral,0.930899,I would like to request access to the SFDC CRM...



POSITIVE — HIGH CONFIDENCE EXAMPLES


,sentiment_label,sentiment_confidence,description
216,Positive,0.951049,"Hey IT Team, Can we set up access to Adobe Fre..."
177,Positive,0.936315,Hi IT Team! Hope you're having a great day. Co...
273,Positive,0.915344,"Hello IT Team, Could you please grant me acces..."
401,Positive,0.909368,"Hi IT Team, I hope this message finds you well..."
189,Positive,0.902889,Hi IT! Hope you're having a great day! It seem...



LOW-CONFIDENCE EXAMPLES


,sentiment_label,sentiment_confidence,description
498,Negative,0.448063,"Hey IT Team, I hope you're doing well! I'm rea..."
37,Neutral,0.476113,Hello IT Team! \n\nI'm having trouble installi...
211,Neutral,0.477141,"Hi IT, I hope you're doing well! If possible, ..."
215,Neutral,0.486725,"Hey IT Team, I hope you're doing well! I'm rea..."
454,Neutral,0.490022,Notification from Acme Cloud Services\n\nAcme ...
389,Neutral,0.490390,Notification from Network Monitoring System\n\...
296,Neutral,0.490818,Notification from Acme Cloud Services\n\nAcme ...
57,Negative,0.491311,"Hi IT Team, I'm experiencing an issue with acc..."
344,Negative,0.492843,"Hey IT Team, I hope you're doing well! I'm rea..."
467,Negative,0.494654,"Hi team, We've encountered an issue with setti..."


In [8]:
# ============================================================
# Step 7 — Confidence-Based Pseudo-Label Quality
# ============================================================

thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]

rows = []

for threshold in thresholds:
    subset = df[df["sentiment_confidence"] >= threshold]

    distribution = (
        subset["sentiment_label"]
        .value_counts()
        .to_dict()
    )

    rows.append(
        {
            "threshold": threshold,
            "retained_rows": len(subset),
            "retained_percent": round(
                len(subset) / len(df) * 100, 2
            ),
            "negative": distribution.get("Negative", 0),
            "neutral": distribution.get("Neutral", 0),
            "positive": distribution.get("Positive", 0),
        }
    )

confidence_summary = pd.DataFrame(rows)

display(confidence_summary)

,threshold,retained_rows,retained_percent,negative,neutral,positive
0,0.5,483,96.6,208,238,37
1,0.6,399,79.8,176,194,29
2,0.7,281,56.2,132,134,15
3,0.8,145,29.0,57,78,10
4,0.9,26,5.2,0,21,5


In [9]:
# Sentiment distribution among high-confidence predictions

high_confidence = df[
    df["sentiment_confidence"] >= 0.70
]

print(
    f"Rows with confidence >= 0.70: "
    f"{len(high_confidence):,} / {len(df):,}"
)

display(
    high_confidence["sentiment_label"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

Rows with confidence >= 0.70: 281 / 500


,sentiment,count
0,Neutral,134
1,Negative,132
2,Positive,15


In [10]:
# ============================================================
# Step 8 — Final Sentiment Analysis Dataset
# ============================================================

sentiment_output = df[
    [
        "id",
        "description",
        "sentiment_label",
        "sentiment_confidence",
    ]
].copy()

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "sentimental_help_analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_PATH = (
    OUTPUT_DIR
    / "it_helpdesk_sentiment_analysis.csv"
)

sentiment_output.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Sentiment analysis output saved.")
print(f"Path: {OUTPUT_PATH}")
print(f"Rows: {len(sentiment_output):,}")

display(sentiment_output.head(10))

Sentiment analysis output saved.
Path: C:\Users\danie\supportmind-ai\data\processed\sentimental_help_analysis\it_helpdesk_sentiment_analysis.csv
Rows: 500


,id,description,sentiment_label,sentiment_confidence
0,1aiu3lrqi,Hey IT! Our network printer keeps disconnectin...,Negative,0.864992
1,kz5mjjpox,This is a follow-up to your previous request #...,Negative,0.644089
2,86eza0fwq,Hey team! :wave: We're experiencing some inter...,Negative,0.571593
3,jtw509e3n,We need to set up a new Google Calendar for te...,Neutral,0.897825
4,tso616mbn,Jordan Smith is part of the Project Management...,Neutral,0.878147
5,6bqwkmxi1,Email client fails to sync with server due to ...,Negative,0.804562
6,cythbyl5p,Notification from Acme IT\n\nAcme IT Image\n\n...,Neutral,0.851358
7,u43q44ro8,I am unable to access my email account as I ha...,Negative,0.645146
8,nay58n9jf,"Hello IT Team, I've been trying to access and ...",Neutral,0.505374
9,2apbmy56h,"Hello IT Team, I've noticed that the group pol...",Negative,0.696259


In [11]:
# ============================================================
# Step 9 — Final Sentiment Validation
# ============================================================

print("Sentiment Analysis Summary")
print("=" * 60)

print(f"Total tickets analyzed : {len(sentiment_output):,}")
print(f"Negative              : {(sentiment_output['sentiment_label'] == 'Negative').sum():,}")
print(f"Neutral               : {(sentiment_output['sentiment_label'] == 'Neutral').sum():,}")
print(f"Positive              : {(sentiment_output['sentiment_label'] == 'Positive').sum():,}")

print(
    f"\nMean confidence       : "
    f"{sentiment_output['sentiment_confidence'].mean():.4f}"
)

print(
    f"High-confidence (>=0.70): "
    f"{(sentiment_output['sentiment_confidence'] >= 0.70).sum():,}"
)

print("\nSaved artifact:")
print(OUTPUT_PATH)

print("\nStatus: SENTIMENT ANALYSIS COMPLETE")

Sentiment Analysis Summary
Total tickets analyzed : 500
Negative              : 215
Neutral               : 246
Positive              : 39

Mean confidence       : 0.7170
High-confidence (>=0.70): 281

Saved artifact:
C:\Users\danie\supportmind-ai\data\processed\sentimental_help_analysis\it_helpdesk_sentiment_analysis.csv

Status: SENTIMENT ANALYSIS COMPLETE
